# Vamos aprender a trabalhar com PDF usando o Python

- Regra geral: PDF foi feito justamente para bloquear muita coisa, então não é fácil "brincar" com um pdf
- Mesmo assim, Python tem várias bibliotecas que vão nos ajudar, vamos focar em 2:
    - PyPDF2
    - Tabula
- Ler e extrair informações de um PDF a gente consegue fazer.
- Escrever e Editar, aí já é outra história

### Para os nossos exemplos, vamos avaliar o Release de Resultados do 3º e 4º Trimestre de 2020 da Magazine Luiza

#### 1º Objetivo: Queremos conseguir separar apenas o DRE do Release de Resultados (Página 14) para enviar para a Diretoria, como fazemos?
    - Separar as páginas de um pdf

In [18]:
import PyPDF2 as pyf
from pathlib import Path

nome = "MGLU_ER_3T20_POR.pdf"
arquivo_pdf = pyf.PdfReader(nome)

for i, pagina in enumerate(arquivo_pdf.pages):
    num_pagina = i + 1
    novo_pdf = pyf.PdfWriter()
    novo_pdf.add_page(pagina)
    with Path(f"paginas/Arquivo Pagina {num_pagina}.pdf").open(mode="wb") as arquivo:
        novo_pdf.write(arquivo)

#### 2º Objetivo: Com o Release de Resultados já separado página por página, queremos incluir apenas as Páginas de Destaque (Página 1), DRE (Página 14) e Balanço (Página 16).
    - Juntar vários pdfs em 1

In [19]:
import PyPDF2 as pyf
from pathlib import Path

num_paginas = [1, 14, 16]

novo_arquivo = pyf.PdfWriter()
for num in num_paginas:
    pagina_pdf = pyf.PdfReader(f"paginas/Arquivo Pagina {num}.pdf")
    novo_arquivo.add_page(pagina_pdf.pages[0])

    
with Path(f"Consolidado.pdf").open(mode="wb") as arquivo:
    novo_arquivo.write(arquivo)

### Extra: Para adicionar todas as páginas de 2 pdfs

In [20]:
pdf_mesclado = pyf.PdfMerger()

pdf_mesclado.append("MGLU_ER_3T20_POR.pdf")
pdf_mesclado.append("MGLU_ER_4T20_POR.pdf")

with Path(f"Mesclado.pdf").open(mode="wb") as arquivo:
    pdf_mesclado.write(arquivo)

# Funcionalidades que podem ser úteis:

- Inserir arquivo no meio do outro
- Quero colocar dentro do Resultado do 4T20 os destaques do 3T20 para poder comparar os 2 dentro do mesmo relatório

In [21]:
pdf_mesclado = pyf.PdfMerger()

pdf_mesclado.append("MGLU_ER_4T20_POR.pdf")
pdf_mesclado.merge(1, "paginas/Arquivo Pagina 1.pdf")

with Path(f"Relatório 2 Trimestres.pdf").open(mode="wb") as arquivo:
    pdf_mesclado.write(arquivo)

- Rodar Página

In [22]:
arquivo_pdf_original = pyf.PdfReader("MGLU_ER_3T20_POR.pdf")

novo_arquivo = pyf.PdfWriter()

for pagina in arquivo_pdf_original.pages:
    pagina.rotate(90)
    novo_arquivo.add_page(pagina)

with Path(f"Paginas Rotacionadas.pdf").open(mode="wb") as arquivo:
    novo_arquivo.write(arquivo)


# Trabalhando com Textos e Informações Dentro do PDF

#### 1º Objetivo: Quero identificar como foram as Despesas com Vendas da MGLU
    - Pegar texto da página e identificar onde está essa informação

In [23]:
arquivo = pyf.PdfReader("MGLU_ER_3T20_POR.pdf")

qtde_paginas = len(arquivo.pages)
print(qtde_paginas)

metadados_arquivo = arquivo.metadata
print(metadados_arquivo)

24
{'/Title': 'DESEMPENHO FINANCEIRO CONSOLIDADO', '/Author': 'an_rezende', '/Subject': 'Receita Bruta', '/Creator': 'Microsoft® Office Word 2007', '/CreationDate': "D:20201109183121-03'00'", '/ModDate': "D:20201109183121-03'00'", '/Producer': 'Microsoft® Office Word 2007'}


In [24]:
texto_referencia = "| Despesas com Vendas"

for i, pagina in enumerate(arquivo.pages):
    texto_pagina = pagina.extract_text()
    if texto_referencia in texto_pagina:
        print("Numero Pagina: ", i+1)
        texto_analisar = texto_pagina

Numero Pagina:  10


In [25]:
print(texto_analisar)

Divulgação de Resultados  
3T20 
10 
 
  
 
| Despesas Operacionais  
 
R$ milhões  3T20  
Ajustado   % RL  3T19  
Ajustado   % RL   Var(%)  9M20  
Ajustado   % RL  9M19  
Ajustado   % RL   Var(%)  
  Despesas com Vendas   (1.432,6)  -17,2%  (890,0)  -18,3%  61,0%  (3.487,2)  -18,2%  (2.309,1)  -17,1%  51,0%  
  Despesas Gerais e Administrativas   (240,7)  -2,9%  (207,1)  -4,3%  16,2%  (617,3)  -3,2%  (498,2)  -3,7%  23,9%  
 Subtotal      (1.673,3)  -20,1%      (1.097,1)  -22,6%  52,5%      (4.104,5)  -21,5%      (2.807,4)  -20,8%  46,2%  
  Perdas em Liquidação Duvidosa   (25,4)  -0,3%  (20,2)  -0,4%  25,4%  (84,5)  -0,4%  (45,8)  -0,3%  84,3%  
  Outras Receitas Operacionais, Líquidas   15,2  0,2%  15,3  0,3%  -0,6%  41,0  0,2%  44,0  0,3%  -6,8%  
  Total de Despesas Operacionais       (1.683,5)  -20,3%      (1.102,0)  -22,7%  52,8%      (4.148,0)  -21,7%      (2.809,2)  -20,8%  47,7%  
 
| Despesas com Vendas  
 
No 3T20, as despesas com vendas totalizaram R$1.432,6 milhões, equiv

In [26]:
posicao_inicial = texto_analisar.find(texto_referencia)
posicao_final = texto_analisar.find("|", posicao_inicial + 1)

texto_final = texto_analisar[posicao_inicial:posicao_final]
print(texto_final)

| Despesas com Vendas  
 
No 3T20, as despesas com vendas totalizaram R$1.432,6 milhões, equivalentes a 17,2% da receita líquida, 1,1 p.p. menor que no 
3T19 , principalmente devido ao forte crescimento das vendas . Vale ressaltar que a Companhia conseguiu diluir as despesas com 
vendas m esmo investi ndo em maior nível de serviço,  especialmente em  atendimento e logística.  
 
Nos 9M20, as despesas com vendas totalizaram R$3.487,2 milhões, equivalentes a 18,2% da receita líquida (+1,1 p.p. versus  os 
9M19).  
 



#### 2º Objetivo: Quero analisar o DRE (sem ajuste - Página 5)
    - Para ler tabelas em pdf, use o tabula (é ninja)
    
    - Cuidado 1: Instale o tabula-py (não instale o tabula). Se instalar o tabula errado, desinstale ele, instale o tabula-py, desinstale o tabula-py e instale novamente o tabula-py. Reinicie o kernel do Jupyter após isso
    
    - Cuidado 2: Tem que ter o java instalado no seu computador (depois de instalar, reinicie o computador)

In [27]:
import tabula

tabelas = tabula.read_pdf("MGLU_ER_3T20_POR.pdf", pages=5)
df_resultado = tabelas[0]
# excluir linhas totalmente vazias
df_resultado = df_resultado.dropna(how="all", axis=0)
# excluir colunas totalmente vazias
df_resultado = df_resultado.dropna(how="all", axis=1)
df_resultado.columns = df_resultado.iloc[0]
df_resultado = df_resultado.iloc[1:]
df_resultado = df_resultado.reset_index(drop=True)
display(df_resultado)

AttributeError: module 'tabula' has no attribute 'read_pdf'

In [29]:
import tabula
print(tabula.__file__)

c:\Users\guiho\AppData\Local\Programs\Python\Python312\Lib\site-packages\tabula\__init__.py
